# SigLIP2 product recognition
This notebook evaluates SigLIP2 using the same saved images and
candidate descriptions as CLIP.

In [1]:
%pip install -q transformers pillow pandas scikit-learn safetensors sentencepiece

Output zip file from 04a_clip.ipynb is needed here to upload.


In [2]:
from pathlib import Path
import json
import zipfile
import hashlib
import time
import platform

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image
from google.colab import files

assert torch.cuda.is_available(), "Select a T4 GPU runtime first."

project = Path("/content/product_recognition")
project.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
assert len(zip_names) == 1, "Upload the CLIP backup ZIP."

with zipfile.ZipFile(zip_names[0]) as archive:
    for name in archive.namelist():
        destination = (project / name).resolve()
        assert destination.is_relative_to(project.resolve())
    archive.extractall(project)

del uploaded

with open(project / "results/candidate_labels.json") as file:
    products = json.load(file)

image_manifest = pd.read_csv(
    project / "results/image_manifest.csv"
)

assert len(products) == 10
assert len(image_manifest) == 50

# Verify the saved images
for row in image_manifest.itertuples(index=False):
    path = project / row.image_file
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row.sha256

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("Candidates:", len(products))
print("Verified images:", len(image_manifest))
print("Shared images and descriptions restored.")

Saving pair4_clip_backup.zip to pair4_clip_backup.zip
GPU: Tesla T4
Candidates: 10
Verified images: 50
Shared images and descriptions restored.


In [ ]:
from huggingface_hub import model_info
from transformers import AutoProcessor, AutoModel

model_id = "google/siglip2-base-patch16-224"
# revision = model_info(model_id).sha
# The checkpoint revision recorded in the original SigLIP2 experiment.
revision = "75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2"
start = time.perf_counter()

processor = AutoProcessor.from_pretrained(
    model_id,
    revision=revision,
)

model = AutoModel.from_pretrained(
    model_id,
    revision=revision,
    dtype=torch.float32,
).to(device)

model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - start

run_config = {
    "model": "SigLIP2",
    "model_id": model_id,
    "revision": revision,
    "precision": "float32",
    "load_seconds": load_seconds,
    "parameters": sum(p.numel() for p in model.parameters()),
    "candidate_order": list(products),
    "candidate_descriptions": list(products.values()),
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "evaluation": "fixed ten-candidate zero-shot classification",
}

with open(project / "results/siglip2_config.json", "w") as file:
    json.dump(run_config, file, indent=2)

print("SigLIP2 loaded.")
print("Revision:", revision)
print("Parameters:", run_config["parameters"])
print("Load seconds:", round(load_seconds, 2))

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP2 loaded.
Revision: 75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2
Parameters: 375187970
Load seconds: 27.36


### Predictions and measurements
SigLIP2 ranks the same ten descriptions for each saved image.
Its processor uses the checkpoint's image settings and maximum-length
text padding.


In [4]:
candidate_ids = list(products)
candidate_texts = list(products.values())

prediction_records = []
score_records = []

# Warm up
first_path = project / image_manifest.iloc[0]["image_file"]

with Image.open(first_path) as opened:
    warmup_image = opened.convert("RGB")

warmup_inputs = processor(
    text=candidate_texts,
    images=warmup_image,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
).to(device)

with torch.inference_mode():
    for _ in range(3):
        warmup_output = model(**warmup_inputs)

torch.cuda.synchronize()
del warmup_output, warmup_inputs

for row in image_manifest.itertuples(index=False):
    with Image.open(project / row.image_file) as opened:
        image = opened.convert("RGB")

    inputs = processor(
        text=candidate_texts,
        images=image,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    ).to(device)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model(**inputs)

    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3

    scores = output.logits_per_image[0].float().cpu().numpy()
    assert np.isfinite(scores).all()

    ranking = np.argsort(-scores)
    ranked_ids = [candidate_ids[index] for index in ranking]
    true_rank = ranked_ids.index(row.product) + 1

    prediction_records.append({
        "model": "SigLIP2",
        "image_id": row.image_id,
        "product": row.product,
        "condition": row.condition,
        "image_file": row.image_file,
        "predicted_product": ranked_ids[0],
        "top3_products": " | ".join(ranked_ids[:3]),
        "true_product_rank": true_rank,
        "correct_top1": int(true_rank == 1),
        "correct_top3": int(true_rank <= 3),
        "top1_score": float(scores[ranking[0]]),
        "score_margin": float(
            scores[ranking[0]] - scores[ranking[1]]
        ),
        "forward_seconds": seconds,
        "peak_allocated_gpu_gib": peak_gib,
    })

    for rank, index in enumerate(ranking, start=1):
        score_records.append({
            "model": "SigLIP2",
            "image_id": row.image_id,
            "candidate": candidate_ids[index],
            "rank": rank,
            "score": float(scores[index]),
        })

    del output, inputs

predictions = pd.DataFrame(prediction_records)
candidate_scores = pd.DataFrame(score_records)

assert len(predictions) == 50
assert len(candidate_scores) == 500

predictions.to_csv(
    project / "outputs/siglip2_predictions.csv",
    index=False,
)
candidate_scores.to_csv(
    project / "outputs/siglip2_candidate_scores.csv",
    index=False,
)

condition_summary = predictions.groupby(
    "condition", sort=False
).agg(
    images=("image_id", "count"),
    top1_accuracy=("correct_top1", "mean"),
    top3_accuracy=("correct_top3", "mean"),
    mean_forward_seconds=("forward_seconds", "mean"),
)

condition_summary.to_csv(
    project / "results/siglip2_condition_summary.csv"
)

overall = pd.DataFrame([{
    "model": "SigLIP2",
    "images": len(predictions),
    "source_photographs": predictions["product"].nunique(),
    "top1_accuracy": predictions["correct_top1"].mean(),
    "top3_accuracy": predictions["correct_top3"].mean(),
    "mean_forward_seconds": predictions["forward_seconds"].mean(),
    "median_forward_seconds": predictions["forward_seconds"].median(),
    "peak_allocated_gpu_gib": (
        predictions["peak_allocated_gpu_gib"].max()
    ),
}])

overall.to_csv(
    project / "results/siglip2_overall_summary.csv",
    index=False,
)

run_config["timing_scope"] = (
    "Warmed forward pass: one image and ten candidate texts; "
    "includes both encoders; excludes preprocessing, transfers, "
    "image loading and saving."
)

with open(project / "results/siglip2_config.json", "w") as file:
    json.dump(run_config, file, indent=2)

display(overall.round(4))
display(condition_summary.round(4))

errors = predictions[predictions["correct_top1"] == 0]
print("Top-1 errors:", len(errors))

display(errors[[
    "product",
    "condition",
    "predicted_product",
    "true_product_rank",
]])

print("Saved SigLIP2 predictions, rankings and summaries.")

,model,images,source_photographs,top1_accuracy,top3_accuracy,mean_forward_seconds,median_forward_seconds,peak_allocated_gpu_gib
0,SigLIP2,50,10,0.98,1.0,0.0637,0.0585,1.4302


,images,top1_accuracy,top3_accuracy,mean_forward_seconds
condition,,,,
original,10,1.0,1.0,0.0639
darker,10,1.0,1.0,0.0663
brighter,10,1.0,1.0,0.0605
blurred,10,0.9,1.0,0.0652
centre_crop,10,1.0,1.0,0.0625


Top-1 errors: 1


,product,condition,predicted_product,true_product_rank
23,Mount_Franklin,blurred,Coca_Cola,2


Saved SigLIP2 predictions, rankings and summaries.


### SigLIP2 findings
SigLIP2 correctly classified 49 of 50 images. The blurred Mount Franklin
bottle was classified as Coca-Cola, with Mount Franklin ranked second.
All original images were classified correctly.

CLIP achieved higher top-1 accuracy and used less measured inference
time and GPU memory in this experiment.

In [5]:
import shutil
from google.colab import files

backup = shutil.make_archive(
    "/content/pair4_both_models_backup",
    "zip",
    root_dir=str(project),
)

files.download(backup)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>